# AI Explainer - Colab Workflow (StyleTTS2 + Whisper + SDXL)

This notebook runs the entire middle pipeline on Google Colab:
1. **Voice Generation:** Uses StyleTTS2 (Hyper-realistic Voice Cloning & Synthesis)
2. **Captioning:** Uses faster-whisper for word-level timestamps
3. **Matching:** Matches script segments against your local `clip_index.json`
4. **Images:** Uses Stable Diffusion XL to generate fallbacks for missing clips.

In [6]:
import os
from pathlib import Path

INPUT_DIR = Path('/content/inputs')
AUDIO_DIR = Path('/content/audio')
CAPTIONS_DIR = Path('/content/captions')
IMAGES_DIR = Path('/content/images')
OUTPUT_DIR = Path('/content/output')

for d in [INPUT_DIR, AUDIO_DIR, CAPTIONS_DIR, IMAGES_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directories created!\n')
print('ACTION REQUIRED: Please open the files tab on the left and upload your:')
print('1. script.txt to /content/inputs/')
print('2. clip_index.json to /content/inputs/')
print('3. (Optional) reference.wav to /content/inputs/ for voice cloning.')

Directories created!

ACTION REQUIRED: Please open the files tab on the left and upload your:
1. script.txt to /content/inputs/
2. clip_index.json to /content/inputs/
3. (Optional) reference.wav to /content/inputs/ for voice cloning.


In [2]:
# 1. Install Dependencies
!apt-get install -y espeak-ng  # Required for StyleTTS2 linguistics

# Force numpy down to 1.x to fix the C header ValueError
!pip install -q "numpy<2.0"

# Install styletts2 and other tools WITHOUT the -U flag
# This allows pip to find compatible versions instead of forcing bleeding-edge ones
!pip install -q styletts2 phonemizer faster-whisper diffusers transformers accelerate safetensors huggingface_hub

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
espeak-ng is already the newest version (1.50+dfsg-10ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.5.1 requires huggingface-hub>=0.23.0, but you have huggingface-hub 0.19.4 which is incompatible.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.19.4 which is incompatible.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
espeak-ng is already the newest version (1.50+dfsg-10ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
ERROR: pip's dependency r

In [4]:
# 2. Voice Generation (StyleTTS2)
import torch
import functools

# --- PyTorch 2.6 Fix: Revert torch.load default behavior ---
_original_load = torch.load
@functools.wraps(_original_load)
def _patched_load(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return _original_load(*args, **kwargs)
torch.load = _patched_load
# -----------------------------------------------------------

In [8]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [9]:
# 2. Voice Generation (StyleTTS2)
from styletts2 import tts

print("Loading StyleTTS2...")
my_tts = tts.StyleTTS2()

ref_audio = INPUT_DIR / 'reference.wav'
if ref_audio.exists():
    print(f"\nVoice cloning enabled! Using {ref_audio.name}")
else:
    print("\nNo reference.wav found. Using default voice.")
    ref_audio = None

for txt_file in INPUT_DIR.glob('*.txt'):
    if txt_file.name == 'reference.txt':
        continue

    text = txt_file.read_text().strip()
    if not text:
        continue

    out_wav = AUDIO_DIR / f"{txt_file.stem}.wav"
    print(f"\nGenerating audio for {txt_file.name}...")

    # StyleTTS2 automatically handles chunking and punctuation!
    if ref_audio:
        my_tts.inference(text, target_voice_path=str(ref_audio), output_wav_file=str(out_wav))
    else:
        my_tts.inference(text, output_wav_file=str(out_wav))

    print(f"\u2705 Audio saved to {out_wav}")


Loading StyleTTS2...
Invalid or missing model checkpoint path. Loading default model...
Invalid or missing config path. Loading default config...
Invalid ASR config path. Loading default config...
Invalid ASR model checkpoint path. Loading default model...
Invalid F0 model path. Loading default model...


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


bert loaded
bert_encoder loaded
predictor loaded
decoder loaded
text_encoder loaded
predictor_encoder loaded
style_encoder loaded
diffusion loaded
text_aligner loaded
pitch_extractor loaded
mpd loaded
msd loaded
wd loaded

No reference.wav found. Using default voice.

Generating audio for evil_mortys_master_plan_1.txt...
Cloning default target voice...
177
sˈoʊ | wˈʌt ˈɪf ˈaɪ tˈoʊld jˈu ðˈæt ɹˈɪks ɡɹˈeɪtɪst ˈɛnəmi mˈaɪt nˈɑt bˈi ə vˈɪlən ˈæt ˈɔl ‖ wˈiv sˈin ɹˈɪk ˈænd mˈɔɹti dˈil wˈɪθ sˈʌm kɹˈeɪzi stˈʌf lˈaɪk ðə ˈɛpɪsˌoʊd wˈɛɹ ðˈeɪ ɪŋkˈaʊntɚd ðə kˈaʊnsɪl əv ɹˈɪks bˈʌt ˈivəl mˈɔɹti tˈeɪks ðə kˈeɪk ‖ ðˈɪs ˈɔltɚnɪt vˈɚʒən əv ˈaʊɚ fˈeɪvɚɪt dɪmˈɛnʃən hˈɑpɪŋ dˈuoʊ ˈɪz mˈɔɹ ðən d͡ʒˈʌst ə klˈoʊn ɡˈɔn ɹˈɔŋ ‖ hˈiz bˈɪn mənˈɪpjəlˌeɪtɪŋ ɪvˈɛnts fɹˈʌm bɪhˈaɪnd ðə sˈinz | jˈuzɪŋ hˈɪz ɪntˈɛlɪd͡ʒəns ˈænd kˈʌnɪŋ tə |
sˈoʊ | wˈʌt ˈɪf ˈaɪ tˈoʊld jˈu ðˈæt ɹˈɪks ɡɹˈeɪtɪst ˈɛnəmi mˈaɪt nˈɑt bˈi ə vˈɪlən ˈæt ˈɔl ‖ wˈiv sˈin ɹˈɪk ˈænd mˈɔɹti dˈil wˈɪθ sˈʌm kɹˈeɪzi stˈʌf lˈaɪk ðə ˈɛpɪsˌoʊd wˈɛɹ ðˈeɪ ɪŋkˈaʊntɚd 

In [10]:
# 3. Captioning (Faster-Whisper)
from faster_whisper import WhisperModel
import json

print('Loading Whisper...')
model = WhisperModel('small', device='cuda', compute_type='float16')

for wav_file in AUDIO_DIR.glob('*.wav'):
    print(f'Captioning {wav_file.name}...')
    segments, info = model.transcribe(str(wav_file), word_timestamps=True)

    cap_data = {'audio_file': str(wav_file), 'segments': []}
    for i, seg in enumerate(segments):
        words = [{'word': w.word, 'start': w.start, 'end': w.end} for w in seg.words]
        cap_data['segments'].append({
            'id': i,
            'text': seg.text.strip(),
            'start': seg.start,
            'end': seg.end,
            'words': words
        })

    out_cap = CAPTIONS_DIR / f"{wav_file.stem}.json"
    with open(out_cap, 'w') as f:
        json.dump(cap_data, f, indent=2)
    print(f"Captions saved to {out_cap}")

Loading Whisper...
Captioning evil_mortys_master_plan_1.wav...
Captions saved to /content/captions/evil_mortys_master_plan_1.json


In [11]:
# 4. Matching
import re

clip_index_path = INPUT_DIR / 'clip_index.json'
clips = []
if clip_index_path.exists():
    clips = json.loads(clip_index_path.read_text()).get('clips', [])
    print(f"Loaded {len(clips)} clips from clip_index.json")
else:
    print("No clip_index.json found. All segments will use AI images.")

def extract_keywords(text):
    words = re.sub(r'[^a-z0-9\s]', '', text.lower()).split()
    stops = {'the', 'is', 'at', 'which', 'on', 'in', 'a', 'an', 'of', 'to', 'and', 'or', 'for'}
    return {w for w in words if w not in stops and len(w) > 1}

for cap_file in CAPTIONS_DIR.glob('*.json'):
    cap_data = json.loads(cap_file.read_text())
    manifest = {'audio_file': cap_data['audio_file'], 'segments': []}

    for seg in cap_data['segments']:
        best_clip = None
        best_score = 0
        seg_kw = extract_keywords(seg['text'])

        for c in clips:
            clip_kw = extract_keywords(c.get('action', '')) | set(c.get('tags', []))
            score = len(seg_kw & clip_kw)
            if score > best_score:
                best_score = score
                best_clip = c

        entry = {
            'id': seg['id'],
            'text': seg['text'],
            'start': seg['start'],
            'end': seg['end'],
            'words': seg['words']
        }

        if best_clip:
            entry['visual_type'] = 'clip'
            entry['visual_source'] = best_clip['filename']
            entry['clip_start'] = 0.0
        else:
            # Fallback to random clip if score is exactly 0
            import random
            random_clip = random.choice(clips) if clips else {'filename': '__generic_broll__'}
            entry['visual_type'] = 'clip'
            entry['visual_source'] = random_clip.get('filename', '__generic_broll__')
            entry['clip_start'] = 0.0

        manifest['segments'].append(entry)

    out_manifest = OUTPUT_DIR / f"manifest_{cap_file.stem}.json"
    with open(out_manifest, 'w') as f:
        json.dump(manifest, f, indent=2)


Loaded 303 clips from clip_index.json
Manifest saved to /content/output/manifest_evil_mortys_master_plan_1.json


In [ ]:
# 6. Package and Zip Outputs
import shutil

print('Zipping outputs...')
# Zip the whole content directory except the inputs and huge repos to save time
os.makedirs('/content/zip_staging', exist_ok=True)
for folder in ['audio', 'captions', 'images', 'output']:
    src = f'/content/{folder}'
    dst = f'/content/zip_staging/{folder}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)

shutil.make_archive('/content/explainer_outputs', 'zip', '/content/zip_staging')
print('\n\u2705 Done! Please download /content/explainer_outputs.zip')

# Optional: Automatically trigger download in browser
try:
    from google.colab import files
    files.download('/content/explainer_outputs.zip')
except ImportError:
    pass